# FORCE-OPT for Aviation: Safety Monitoring of Airport Surface Movement

Adaptation of [FORCE-OPT](https://arxiv.org/abs/2507.22389) (Chakraborty et al., 2025) for use with
[Amelia-TF](https://github.com/AmeliaCMU/AmeliaTF) (Navarro et al., 2024), a transformer-based
trajectory forecasting model for airport surface movement.

---

## Key Adaptations from Automotive → Aviation

| Aspect | FORCE-OPT (nuScenes) | This Adaptation (Amelia-TF) |
|---|---|---|
| Domain | Road driving | Airport surface (taxiways, runways, ramps) |
| Agents | Cars, pedestrians, cyclists | Aircraft, ground vehicles, tugs |
| State space | 2D position (x, y) meters | 2D position (x, y) from lat/lon projection |
| Predictor output | GMM per timestep | GMM per timestep (same structure) |
| Separation standards | ~2 m collision buffer | FAA/ICAO separation minima (60–90 m on taxiways) |
| Worst-case dynamics | Dubins car (bounded accel/yaw) | Taxiing aircraft (bounded speed/turn on surface) |
| Safety events | Collision with other vehicle | Runway incursion, taxiway conflict, ramp collision |
| OOD scenarios | Different city | Unseen airport topology |

In [ ]:
import numpy as np
from scipy.stats import chi2
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from enum import Enum
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Ellipse, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

---
## 1. Aviation-Specific Data Structures

We define data structures tailored to airport surface operations, including
aircraft agent types, airport zone types, and FAA/ICAO separation standards.

In [ ]:
class AgentType(Enum):
    """Types of agents on airport surface, as categorized by Amelia-48."""
    AIRCRAFT = 'aircraft'
    GROUND_VEHICLE = 'ground_vehicle'
    UNKNOWN = 'unknown'


class ZoneType(Enum):
    """Airport surface zones with different safety separation requirements."""
    RUNWAY = 'runway'          # Most safety-critical
    TAXIWAY = 'taxiway'        # Standard taxi operations
    RAMP = 'ramp'              # Gate/apron area
    HOLDING_AREA = 'holding'   # Run-up pad / hold short area
    GENERAL = 'general'


# FAA/ICAO-inspired separation standards (meters) by zone and wake category
# These are approximate and conservative values for surface movement
SEPARATION_STANDARDS = {
    ZoneType.RUNWAY: {
        'default': 90.0,          # ~300 ft, standard runway separation
        'same_direction': 60.0,   # Reduced when following in same direction
    },
    ZoneType.TAXIWAY: {
        'default': 45.0,          # ~150 ft, taxiway separation
        'wing_tip': 25.0,         # Wing-tip clearance
    },
    ZoneType.RAMP: {
        'default': 15.0,          # ~50 ft, ramp area (slower, tighter)
        'wing_tip': 7.5,
    },
    ZoneType.HOLDING_AREA: {
        'default': 75.0,
    },
    ZoneType.GENERAL: {
        'default': 30.0,
    },
}


@dataclass
class GMMComponent:
    """Single Gaussian component from the Amelia-TF GMM decoder output.
    
    Amelia-TF's trajectory decoder produces M modes, each with a mean trajectory
    μ_x ∈ R^D, covariance Σ_x ∈ R^(D×D), and confidence score ρ_m.
    This class represents one mode at one timestep.
    """
    mean: np.ndarray        # (2,) predicted (x, y) position
    covariance: np.ndarray  # (2, 2) covariance matrix
    weight: float           # ρ_m mixing weight


@dataclass
class GMMDistribution:
    """GMM distribution at a single future timestep, as output by Amelia-TF."""
    components: List[GMMComponent]
    
    @property
    def K(self) -> int:
        return len(self.components)
    
    @property
    def weights(self) -> np.ndarray:
        return np.array([c.weight for c in self.components])


@dataclass
class AircraftAgent:
    """An aircraft or ground vehicle on the airport surface.
    
    Attributes:
        agent_id: Unique identifier (e.g., callsign or track ID from SWIM SMES).
        agent_type: Aircraft, ground vehicle, or unknown.
        history: (H, 2) array of observed (x, y) positions over history horizon H.
        wingspan: Physical wingspan in meters (affects separation calculations).
        current_speed: Current ground speed in m/s.
        current_heading: Current heading in radians.
    """
    agent_id: str
    agent_type: AgentType
    history: np.ndarray
    wingspan: float = 36.0      # Default: Boeing 737 class (~118 ft)
    current_speed: float = 8.0  # Default: ~15 knots taxi speed
    current_heading: float = 0.0
    
    @property
    def current_position(self) -> np.ndarray:
        return self.history[-1]


@dataclass
class SurfacePlan:
    """Planned surface movement trajectory for the ego aircraft.
    
    This could come from a taxi route planner, CDTI prediction, or controller
    instruction. Represents the ego aircraft's intended path across the surface.
    
    Attributes:
        positions: (T, 2) array of planned (x, y) positions.
        agent: The ego aircraft executing this plan.
        zone_at_step: Optional list of zone types at each timestep.
    """
    positions: np.ndarray
    agent: AircraftAgent
    zone_at_step: Optional[List[ZoneType]] = None
    
    def get_zone(self, t: int) -> ZoneType:
        if self.zone_at_step and t < len(self.zone_at_step):
            return self.zone_at_step[t]
        return ZoneType.GENERAL
    
    def get_separation(self, t: int) -> float:
        """Get the required separation minimum at timestep t."""
        zone = self.get_zone(t)
        base = SEPARATION_STANDARDS.get(zone, SEPARATION_STANDARDS[ZoneType.GENERAL])
        return base['default']

---
## 2. Amelia-TF Output Adapter

This adapter converts Amelia-TF's raw PyTorch output tensors into the `GMMDistribution`
objects that FORCE-OPT operates on. Amelia-TF's decoder outputs per-mode means, covariances,
and confidence scores for each agent across T future timesteps.

In [ ]:
class AmeliaTFAdapter:
    """Adapter to convert Amelia-TF model outputs to FORCE-OPT GMMDistributions.
    
    Amelia-TF's trajectory decoder outputs a GMM for each agent:
      - mu:    (N_agents, M, T, D) predicted mean positions per mode per timestep
      - sigma: (N_agents, M, T, D, D) predicted covariances
      - rho:   (N_agents, M) mode confidence/mixing weights (sum to 1 per agent)
    
    where N_agents is the number of scored agents in the scene, M is the number of
    GMM modes, T is the prediction horizon, and D=2 for (x, y) positions.
    
    The Amelia-48 dataset uses relative coordinates (meters from a reference point),
    projected from lat/lon. This adapter handles both raw tensor and numpy inputs.
    """
    
    def __init__(self, min_eigenvalue: float = 1e-4):
        """Args:
            min_eigenvalue: Floor for covariance eigenvalues to ensure positive-definiteness.
        """
        self.min_eigenvalue = min_eigenvalue
    
    def _ensure_psd(self, cov: np.ndarray) -> np.ndarray:
        """Ensure a 2x2 covariance matrix is positive semi-definite.
        
        Amelia-TF may occasionally produce near-singular covariances,
        especially for stopped/parked aircraft. We regularize these.
        """
        # Symmetrize
        cov = (cov + cov.T) / 2.0
        eigvals, eigvecs = np.linalg.eigh(cov)
        eigvals = np.maximum(eigvals, self.min_eigenvalue)
        return eigvecs @ np.diag(eigvals) @ eigvecs.T
    
    def convert_single_agent(
        self,
        mu: np.ndarray,
        sigma: np.ndarray,
        rho: np.ndarray,
    ) -> List[GMMDistribution]:
        """Convert one agent's Amelia-TF output to a list of per-timestep GMMs.
        
        Args:
            mu: (M, T, 2) predicted mean positions for each mode and timestep.
            sigma: (M, T, 2, 2) predicted covariance matrices.
            rho: (M,) mode mixing weights (confidence scores).
        
        Returns:
            List of T GMMDistribution objects.
        """
        M, T, D = mu.shape
        assert D == 2, f"Expected 2D positions, got D={D}"
        assert sigma.shape == (M, T, 2, 2)
        
        # Normalize weights to sum to 1
        weights = rho / (rho.sum() + 1e-10)
        
        gmms = []
        for t in range(T):
            components = []
            for m in range(M):
                cov = self._ensure_psd(sigma[m, t])
                components.append(GMMComponent(
                    mean=mu[m, t].copy(),
                    covariance=cov,
                    weight=float(weights[m]),
                ))
            gmms.append(GMMDistribution(components))
        
        return gmms
    
    def convert_batch(
        self,
        mu_batch: np.ndarray,
        sigma_batch: np.ndarray,
        rho_batch: np.ndarray,
    ) -> Dict[int, List[GMMDistribution]]:
        """Convert a batch of Amelia-TF outputs for all agents in a scene.
        
        Args:
            mu_batch: (N, M, T, 2) batch of mean predictions.
            sigma_batch: (N, M, T, 2, 2) batch of covariance predictions.
            rho_batch: (N, M) batch of mixing weights.
        
        Returns:
            Dictionary mapping agent index → list of T GMMDistributions.
        """
        N = mu_batch.shape[0]
        return {
            i: self.convert_single_agent(
                mu_batch[i], sigma_batch[i], rho_batch[i]
            )
            for i in range(N)
        }


# Quick test
adapter = AmeliaTFAdapter()
M, T, D = 5, 20, 2
fake_mu = np.random.randn(M, T, D) * 50
fake_sigma = np.zeros((M, T, D, D))
for m in range(M):
    for t in range(T):
        A = np.random.randn(D, D) * 5
        fake_sigma[m, t] = A @ A.T + np.eye(D) * 0.1
fake_rho = np.array([0.4, 0.25, 0.2, 0.1, 0.05])

gmms = adapter.convert_single_agent(fake_mu, fake_sigma, fake_rho)
print(f"Converted: {len(gmms)} timesteps, {gmms[0].K} modes each")
print(f"Weights at t=0: {gmms[0].weights}")

---
## 3. Core FORCE-OPT Engine (Aviation-Adapted)

The core convex optimization and conformal prediction machinery from FORCE-OPT remains
unchanged — the math is identical since both domains use 2D GMMs. The key changes are:

- **Separation-aware safety check**: Instead of simple point-in-ellipse collision, we use
  zone-dependent separation minima (FAA/ICAO standards)
- **Aviation worst-case FRS**: Taxiing aircraft dynamics (bounded ground speed, limited turn rate)
- **Multi-agent scene evaluation**: Check ego plan against all nearby agents simultaneously

In [ ]:
def solve_frs_convex_optimization(
    gmm: GMMDistribution,
    tau: float = 0.95,
) -> np.ndarray:
    """Solve the convex optimization (Eq. 6 of FORCE-OPT paper) via KKT bisection.
    
    Finds optimal sublevel set thresholds c* for each GMM mode that yield the
    minimum-volume union of ellipses covering at least τ probability mass.
    
    This is unchanged from the original FORCE-OPT — the optimization only depends
    on the GMM structure (weights, eigenvalues), not the application domain.
    """
    K = gmm.K
    weights = gmm.weights
    
    eigenvalues = np.array([
        np.linalg.eigvalsh(c.covariance) for c in gmm.components
    ])  # (K, 2)
    
    area_coeffs = np.pi * np.sqrt(eigenvalues[:, 0] * eigenvalues[:, 1])
    
    def c_from_lambda(lam):
        ratio = 2.0 * area_coeffs / (lam * weights + 1e-30)
        return np.maximum(0.0, -2.0 * np.log(np.clip(ratio, 1e-30, None)))
    
    def coverage_from_lambda(lam):
        c = c_from_lambda(lam)
        return np.dot(weights, 1.0 - np.exp(-c / 2.0))
    
    lam_lo, lam_hi = 1e-6, 1e6
    for _ in range(100):
        lam_mid = (lam_lo + lam_hi) / 2.0
        if coverage_from_lambda(lam_mid) < tau:
            lam_lo = lam_mid
        else:
            lam_hi = lam_mid
        if abs(coverage_from_lambda(lam_mid) - tau) < 1e-12:
            break
    
    return c_from_lambda(lam_hi)


def mahalanobis_energy(x, mean, covariance):
    """V_i(x) = (x - μ_i)^T Σ_i^{-1} (x - μ_i)"""
    diff = x - mean
    return float(diff @ np.linalg.solve(covariance, diff))


def compute_nonconformity_score(x, gmm, c_star):
    """ψ(s, x) = min_i { V_i(x) / c_i* } — the smallest scaling to cover x."""
    scores = []
    for i, comp in enumerate(gmm.components):
        if c_star[i] <= 0:
            continue
        energy = mahalanobis_energy(x, comp.mean, comp.covariance)
        scores.append(energy / c_star[i])
    return min(scores) if scores else float('inf')


def calibrate_conformal(scores, gamma=0.05):
    """Set η as the (1-γ) quantile of non-conformity scores."""
    N = len(scores)
    quantile_level = min(np.ceil((N + 1) * (1 - gamma)) / N, 1.0)
    return float(np.quantile(scores, quantile_level))


def min_distance_to_frs(
    x: np.ndarray,
    gmm: GMMDistribution,
    c_star: np.ndarray,
    eta: float,
) -> float:
    """Compute a normalized 'distance' of point x from the FRS boundary.
    
    Returns a value < 1 if x is inside the FRS, > 1 if outside.
    This is used for separation-aware safety evaluation in aviation.
    """
    ratios = []
    for i, comp in enumerate(gmm.components):
        if c_star[i] <= 0:
            continue
        energy = mahalanobis_energy(x, comp.mean, comp.covariance)
        ratios.append(energy / (eta * c_star[i]))
    return min(ratios) if ratios else float('inf')

---
## 4. Bayesian Confidence Filter (Aviation-Adapted)

The Bayesian filter is especially important in aviation because:
- **Unseen airport topologies** (OOD): Amelia-TF's multi-airport model may be deployed at airports
  not in its training set (e.g., trained on KMDW, deployed at KLGA)
- **Unusual maneuvers**: Go-arounds, emergency stops, runway incursions are rare events
  that the predictor may not model well
- **Weather impacts**: Strong crosswinds, low visibility change surface behavior patterns

In [ ]:
def gmm_likelihood(x, gmm, cov_scale=1.0):
    """Likelihood φ(x; GMM) with scaled covariances."""
    nx = x.shape[0]
    total = 0.0
    for comp in gmm.components:
        scaled_cov = cov_scale * comp.covariance
        diff = x - comp.mean
        sign, logdet = np.linalg.slogdet(scaled_cov)
        mahal = diff @ np.linalg.solve(scaled_cov, diff)
        log_prob = -0.5 * (nx * np.log(2 * np.pi) + logdet + mahal)
        total += comp.weight * np.exp(log_prob)
    return max(total, 1e-300)


class AviationBayesianFilter:
    """Bayesian confidence filter adapted for airport surface operations.
    
    In addition to standard FORCE-OPT belief tracking, this version:
    - Tracks per-agent confidence independently (different aircraft may be more/less
      predictable based on their operational state)
    - Uses a 3-tier confidence → response mapping:
        β > 0.75: Use FORCE-OPT (calibrated predictor)
        0.5 ≤ β ≤ 0.75: Use FORCE-OPT with inflated FRS (+ belief scaling)
        β < 0.5: Switch to worst-case FRS (aviation dynamics model)
    """
    
    def __init__(
        self,
        beta_low: float = 0.3,
        beta_high: float = 1.0,
        tier1_threshold: float = 0.75,  # High confidence
        tier2_threshold: float = 0.50,  # Medium → inflated FRS
    ):
        self.beta_low = beta_low
        self.beta_high = beta_high
        self.tier1_threshold = tier1_threshold
        self.tier2_threshold = tier2_threshold
        self.belief_low = 0.5
        self.belief_high = 0.5
        self.history = [self.expected_beta]
    
    @property
    def expected_beta(self):
        return self.belief_low * self.beta_low + self.belief_high * self.beta_high
    
    @property
    def confidence_tier(self) -> str:
        beta = self.expected_beta
        if beta >= self.tier1_threshold:
            return 'high'   # Trust predictor
        elif beta >= self.tier2_threshold:
            return 'medium' # Inflate FRS
        else:
            return 'low'    # Fall back to worst-case
    
    def update(self, x_obs, gmm, eta):
        """Bayesian update (Eq. 13 of FORCE-OPT paper)."""
        lik_low = gmm_likelihood(x_obs, gmm, cov_scale=(1.0 / self.beta_low) * eta)
        lik_high = gmm_likelihood(x_obs, gmm, cov_scale=(1.0 / self.beta_high) * eta)
        unnorm_low = lik_low * self.belief_low
        unnorm_high = lik_high * self.belief_high
        Z = unnorm_low + unnorm_high
        if Z > 0:
            self.belief_low = unnorm_low / Z
            self.belief_high = unnorm_high / Z
        self.history.append(self.expected_beta)
    
    def reset(self):
        self.belief_low = 0.5
        self.belief_high = 0.5
        self.history = [self.expected_beta]

---
## 5. Worst-Case FRS for Taxiing Aircraft

When the Bayesian filter detects low confidence, we fall back to a physics-based
worst-case FRS. For airport surface operations, we model aircraft using:
- **Taxi speed bounds**: 0–30 knots on taxiways, 0–15 knots on ramp
- **Turn rate limits**: Based on nose-wheel steering limits and taxiway geometry
- **Runway operations**: Up to 160+ knots on takeoff roll

In [ ]:
def compute_aviation_worst_case_frs(
    agent: AircraftAgent,
    dt: float,
    T: int,
    zone: ZoneType = ZoneType.TAXIWAY,
) -> List[Tuple[np.ndarray, float]]:
    """Compute worst-case FRS for a taxiing aircraft.
    
    Uses bounded-dynamics assumptions appropriate for airport surface movement:
    - Taxiway: max speed ~15 m/s (30 kt), max accel ~2 m/s², max yaw ~0.2 rad/s
    - Ramp: max speed ~7.5 m/s (15 kt), max accel ~1 m/s², max yaw ~0.3 rad/s
    - Runway: max speed ~80 m/s (160 kt), max accel ~5 m/s², limited yaw
    
    Returns:
        List of (center, radius) for each of T future timesteps.
    """
    # Zone-dependent dynamics bounds
    bounds = {
        ZoneType.RUNWAY:       {'max_speed': 80.0, 'max_accel': 5.0},
        ZoneType.TAXIWAY:      {'max_speed': 15.0, 'max_accel': 2.0},
        ZoneType.RAMP:         {'max_speed': 7.5,  'max_accel': 1.0},
        ZoneType.HOLDING_AREA: {'max_speed': 5.0,  'max_accel': 1.5},
        ZoneType.GENERAL:      {'max_speed': 15.0, 'max_accel': 2.0},
    }
    b = bounds.get(zone, bounds[ZoneType.GENERAL])
    v0 = agent.current_speed
    pos = agent.current_position.copy()
    
    frs = []
    for t_idx in range(1, T + 1):
        elapsed = t_idx * dt
        # Max distance: integrate speed with max acceleration, capped at max_speed
        v_max = min(v0 + b['max_accel'] * elapsed, b['max_speed'])
        # Distance = integral of speed (trapezoidal)
        max_dist = (v0 + v_max) / 2.0 * elapsed
        max_dist = max(max_dist, 0.0)
        # Circular FRS (aircraft can turn in any direction)
        frs.append((pos.copy(), max_dist))
    
    return frs

---
## 6. Complete Aviation FORCE-OPT Safety Monitor

The main class integrating all components for airport surface safety monitoring.

In [ ]:
class AviationFORCE_OPT:
    """FORCE-OPT safety monitor adapted for airport surface operations.
    
    This monitor takes Amelia-TF predictions for surrounding traffic and evaluates
    whether the ego aircraft's planned surface route is safe, accounting for:
    - Multi-modal trajectory uncertainty via GMM-based FRS
    - Conformal calibration for coverage guarantees
    - FAA/ICAO separation standards by surface zone
    - Bayesian confidence tracking for OOD robustness
    - Worst-case fallback for low-confidence scenarios
    """
    
    def __init__(
        self,
        tau: float = 0.95,
        gamma: float = 0.05,
        use_belief: bool = True,
        use_separation_standards: bool = True,
    ):
        self.tau = tau
        self.gamma = gamma
        self.use_belief = use_belief
        self.use_separation = use_separation_standards
        self.eta = None
        self.adapter = AmeliaTFAdapter()
        self.belief_filters: Dict[str, AviationBayesianFilter] = {}
    
    def calibrate(
        self,
        calibration_data: List[Tuple[GMMDistribution, np.ndarray]],
        verbose: bool = True,
    ):
        """Calibrate using split conformal prediction on held-out Amelia-48 data.
        
        The calibration data should come from the same airport (for single-airport
        models) or across airports (for multi-airport generalization).
        """
        scores = []
        for gmm, x_gt in calibration_data:
            c_star = solve_frs_convex_optimization(gmm, self.tau)
            score = compute_nonconformity_score(x_gt, gmm, c_star)
            scores.append(score)
        
        self.eta = calibrate_conformal(np.array(scores), self.gamma)
        if verbose:
            cov = np.mean(np.array(scores) <= self.eta)
            print(f"Aviation FORCE-OPT calibrated: η={self.eta:.4f}, "
                  f"coverage={cov:.2%}, N={len(scores)}")
    
    def _get_belief_filter(self, agent_id: str) -> AviationBayesianFilter:
        """Get or create a per-agent Bayesian confidence filter."""
        if agent_id not in self.belief_filters:
            self.belief_filters[agent_id] = AviationBayesianFilter()
        return self.belief_filters[agent_id]
    
    def evaluate_plan_safety(
        self,
        ego_plan: SurfacePlan,
        traffic_agents: List[AircraftAgent],
        traffic_gmms: Dict[str, List[GMMDistribution]],
        dt: float = 1.0,
    ) -> dict:
        """Evaluate the safety of an ego aircraft's surface plan against all traffic.
        
        This is the main entry point for the aviation safety monitor. For each
        surrounding agent and each timestep, it checks whether the ego plan
        violates the calibrated FRS expanded by the appropriate separation standard.
        
        Args:
            ego_plan: The ego aircraft's planned surface trajectory.
            traffic_agents: List of surrounding aircraft/vehicles.
            traffic_gmms: Dict mapping agent_id → list of T GMMs from Amelia-TF.
            dt: Timestep interval in seconds.
        
        Returns:
            Dictionary with safety assessment results per agent and overall.
        """
        T = ego_plan.positions.shape[0]
        eta = self.eta if self.eta is not None else 1.0
        
        all_conflicts = []
        agent_results = {}
        
        for agent in traffic_agents:
            if agent.agent_id not in traffic_gmms:
                continue
            
            agent_gmms = traffic_gmms[agent.agent_id]
            T_agent = min(T, len(agent_gmms))
            conflicts = []
            method_used = 'FORCE-OPT'
            
            # Get belief filter for this agent
            if self.use_belief:
                bf = self._get_belief_filter(agent.agent_id)
                tier = bf.confidence_tier
            else:
                bf = None
                tier = 'high'
            
            # Determine effective covariance scale
            if tier == 'low':
                # Fall back to worst-case FRS
                method_used = 'WC-FRS (low confidence)'
                zone = ego_plan.get_zone(0)
                wc_frs = compute_aviation_worst_case_frs(agent, dt, T_agent, zone)
                for t in range(T_agent):
                    center, radius = wc_frs[t]
                    sep = ego_plan.get_separation(t) if self.use_separation else 0
                    dist = np.linalg.norm(ego_plan.positions[t] - center)
                    if dist < radius + sep + agent.wingspan / 2:
                        conflicts.append(t)
            else:
                # Use FORCE-OPT (possibly with belief inflation)
                if tier == 'medium' and bf is not None:
                    effective_scale = eta / bf.expected_beta
                    method_used = 'FORCE-OPT+belief'
                else:
                    effective_scale = eta
                
                for t in range(T_agent):
                    gmm = agent_gmms[t]
                    c_star = solve_frs_convex_optimization(gmm, self.tau)
                    
                    # Check ego position against FRS
                    ego_pos = ego_plan.positions[t]
                    normalized_dist = min_distance_to_frs(
                        ego_pos, gmm, c_star, effective_scale
                    )
                    
                    # Aviation-specific: also check separation standard
                    if self.use_separation:
                        sep = ego_plan.get_separation(t)
                        # Find closest mode center to ego
                        min_center_dist = min(
                            np.linalg.norm(ego_pos - c.mean) for c in gmm.components
                        )
                        separation_violated = min_center_dist < sep
                    else:
                        separation_violated = False
                    
                    if normalized_dist < 1.0 or separation_violated:
                        conflicts.append(t)
            
            agent_results[agent.agent_id] = {
                'conflicts': conflicts,
                'method': method_used,
                'is_safe': len(conflicts) == 0,
                'confidence_tier': tier,
            }
            all_conflicts.extend([(agent.agent_id, t) for t in conflicts])
        
        return {
            'is_safe': len(all_conflicts) == 0,
            'total_conflicts': len(all_conflicts),
            'conflict_details': all_conflicts,
            'per_agent': agent_results,
        }

---
## 7. Synthetic Airport Scenario Generation

We create a realistic airport surface scenario for demonstration, simulating a
taxiway intersection conflict similar to runway incursion scenarios.

In [ ]:
def generate_airport_scenario(
    T: int = 20,
    dt: float = 1.0,
    M: int = 5,
    seed: int = 42,
) -> dict:
    """Generate a synthetic taxiway intersection conflict scenario.
    
    Simulates two aircraft: ego taxiing north along Taxiway A, and a contender
    taxiing east along Taxiway B. Their paths cross at an intersection.
    
    The Amelia-TF predictor produces multi-modal predictions for the contender:
    some modes predict it stopping (holding short), others predict it continuing.
    """
    rng = np.random.RandomState(seed)
    
    # ── Contender aircraft (taxiing east along Taxiway B) ──
    contender_speed = 8.0  # m/s (~15 kt)
    contender_start = np.array([-80.0, 0.0])
    H = 10  # History steps
    contender_history = np.array([
        contender_start + np.array([contender_speed * (i - H) * dt, 0.0])
        for i in range(H)
    ])
    contender = AircraftAgent(
        agent_id='TRAFFIC_1', agent_type=AgentType.AIRCRAFT,
        history=contender_history, wingspan=36.0,
        current_speed=contender_speed, current_heading=0.0,
    )
    
    # Ground truth: contender continues straight through intersection
    gt_trajectory = np.array([
        contender_start + np.array([contender_speed * t * dt, 0.0])
        for t in range(1, T + 1)
    ])
    
    # ── Amelia-TF predictions (M modes) ──
    # Mode 0: Continue straight (high probability)
    # Mode 1: Slow down and stop (hold short)
    # Mode 2: Slight turn right (toward ramp)
    # Mode 3: Accelerate through quickly
    # Mode 4: Slight turn left
    mode_configs = [
        {'heading': 0.0,         'speed': contender_speed,     'weight': 0.40},
        {'heading': 0.0,         'speed': contender_speed*0.2, 'weight': 0.25},
        {'heading': np.deg2rad(-15), 'speed': contender_speed*0.8, 'weight': 0.15},
        {'heading': 0.0,         'speed': contender_speed*1.3, 'weight': 0.12},
        {'heading': np.deg2rad(10),  'speed': contender_speed*0.7, 'weight': 0.08},
    ]
    
    gmms = []
    for t_idx in range(T):
        t = (t_idx + 1) * dt
        components = []
        for m, cfg in enumerate(mode_configs):
            h = cfg['heading']
            s = cfg['speed']
            mean = contender_start + s * t * np.array([np.cos(h), np.sin(h)])
            mean += rng.randn(2) * 0.5  # Small prediction noise
            
            # Covariance: elongated along heading, grows with time
            direction = np.array([np.cos(h), np.sin(h)])
            perp = np.array([-np.sin(h), np.cos(h)])
            R = np.column_stack([direction, perp])
            D = np.diag([(1.0 + 0.5 * t)**2, (0.5 + 0.2 * t)**2])
            cov = R @ D @ R.T + np.eye(2) * 0.01
            
            components.append(GMMComponent(
                mean=mean, covariance=cov, weight=cfg['weight']
            ))
        gmms.append(GMMDistribution(components))
    
    # ── Ego aircraft plans ──
    ego_speed = 7.0  # m/s
    ego_start = np.array([0.0, -120.0])
    ego_history = np.array([
        ego_start + np.array([0.0, ego_speed * (i - H) * dt])
        for i in range(H)
    ])
    ego_agent = AircraftAgent(
        agent_id='EGO', agent_type=AgentType.AIRCRAFT,
        history=ego_history, wingspan=34.0,
        current_speed=ego_speed, current_heading=np.pi/2,
    )
    
    # Safe plan: ego stops well before intersection (hold short at y=-85 (>85m from intersection))
    safe_positions = []
    for t in range(1, T + 1):
        y = min(ego_start[1] + ego_speed * t * dt, -85.0)  # Stop at y=-85 (beyond 60m taxiway separation)
        safe_positions.append([ego_start[0], y])
    safe_plan = SurfacePlan(
        positions=np.array(safe_positions), agent=ego_agent,
        zone_at_step=[ZoneType.TAXIWAY] * T,
    )
    
    # Unsafe plan: ego proceeds through intersection without holding
    unsafe_positions = []
    for t in range(1, T + 1):
        y = ego_start[1] + ego_speed * t * dt
        unsafe_positions.append([ego_start[0], y])
    unsafe_plan = SurfacePlan(
        positions=np.array(unsafe_positions), agent=ego_agent,
        zone_at_step=[ZoneType.TAXIWAY] * T,
    )
    
    return {
        'contender': contender, 'gt_trajectory': gt_trajectory,
        'agent_gmms': gmms, 'ego_agent': ego_agent,
        'safe_plan': safe_plan, 'unsafe_plan': unsafe_plan,
        'T': T, 'dt': dt, 'M': M,
    }


def generate_aviation_calibration_data(n_samples=500, M=5, seed=123):
    """Generate synthetic calibration data mimicking Amelia-TF predictions."""
    rng = np.random.RandomState(seed)
    data = []
    for _ in range(n_samples):
        speed = rng.uniform(2, 15)  # Taxi speeds
        heading = rng.uniform(0, 2 * np.pi)
        t = rng.uniform(1.0, 10.0)
        
        gt = speed * t * np.array([np.cos(heading), np.sin(heading)])
        gt += rng.randn(2) * speed * 0.1
        
        components = []
        weights = rng.dirichlet(np.ones(M) * 2)
        for k in range(M):
            mode_heading = heading + rng.randn() * 0.2
            mean = speed * t * np.array([np.cos(mode_heading), np.sin(mode_heading)])
            mean += rng.randn(2) * speed * 0.08
            direction = np.array([np.cos(mode_heading), np.sin(mode_heading)])
            perp = np.array([-np.sin(mode_heading), np.cos(mode_heading)])
            R = np.column_stack([direction, perp])
            scale = rng.uniform(0.5, 1.5)
            D_mat = np.diag([(scale * speed * 0.15)**2, (scale * speed * 0.08)**2])
            cov = R @ D_mat @ R.T + np.eye(2) * 0.01
            components.append(GMMComponent(mean=mean, covariance=cov, weight=weights[k]))
        data.append((GMMDistribution(components), gt))
    return data


scenario = generate_airport_scenario(T=20, M=5)
cal_data = generate_aviation_calibration_data(n_samples=500, M=5)
print(f"Scenario: T={scenario['T']}, M={scenario['M']}, dt={scenario['dt']}s")
print(f"Calibration data: {len(cal_data)} examples")

---
## 8. Run the Aviation Safety Monitor

In [ ]:
# ── Instantiate and calibrate ──
monitor = AviationFORCE_OPT(
    tau=0.95, gamma=0.05,
    use_belief=True, use_separation_standards=True,
)
monitor.calibrate(cal_data)

# Also create a version without separation standards for comparison
monitor_no_sep = AviationFORCE_OPT(
    tau=0.95, gamma=0.05,
    use_belief=False, use_separation_standards=False,
)
monitor_no_sep.calibrate(cal_data)

In [ ]:
# ── Evaluate both plans ──
traffic_gmms = {scenario['contender'].agent_id: scenario['agent_gmms']}
traffic_agents = [scenario['contender']]

print("=" * 65)
print("AIRPORT SURFACE SAFETY EVALUATION")
print("=" * 65)

for plan_name, plan in [('Hold-Short (safe)', scenario['safe_plan']),
                         ('Proceed-Through (unsafe)', scenario['unsafe_plan'])]:
    print(f"\n--- Ego Plan: {plan_name} ---")
    
    result = monitor.evaluate_plan_safety(
        ego_plan=plan, traffic_agents=traffic_agents,
        traffic_gmms=traffic_gmms, dt=scenario['dt'],
    )
    print(f"  With separation standards:")
    print(f"    Safe: {result['is_safe']}")
    print(f"    Conflicts: {result['total_conflicts']} timesteps")
    for aid, ar in result['per_agent'].items():
        if ar['conflicts']:
            print(f"    Agent {aid}: conflicts at t={ar['conflicts'][:5]}{'...' if len(ar['conflicts'])>5 else ''}")
    
    result2 = monitor_no_sep.evaluate_plan_safety(
        ego_plan=plan, traffic_agents=traffic_agents,
        traffic_gmms=traffic_gmms, dt=scenario['dt'],
    )
    print(f"  Without separation standards:")
    print(f"    Safe: {result2['is_safe']}")
    print(f"    Conflicts: {result2['total_conflicts']} timesteps")

---
## 9. Airport Surface Visualization

Visualizes the taxiway intersection scenario with the contender's predicted FRS,
ground truth trajectory, and both ego plans on an airport surface layout.

In [ ]:
def plot_ellipse(ax, mean, covariance, threshold, color='cyan', alpha=0.3, label=None):
    eigvals, eigvecs = np.linalg.eigh(covariance)
    angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
    width = 2 * np.sqrt(eigvals[1] * max(threshold, 0))
    height = 2 * np.sqrt(eigvals[0] * max(threshold, 0))
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle,
                      facecolor=color, edgecolor=color, alpha=alpha, label=label)
    ax.add_patch(ellipse)


def draw_taxiway_layout(ax):
    """Draw simplified taxiway intersection layout."""
    # Taxiway A (north-south)
    ax.fill_between([-12, 12], -100, 100, color='#d4d4d4', alpha=0.4)
    ax.axvline(x=-12, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(x=12, color='gray', linewidth=0.5, linestyle='--')
    
    # Taxiway B (east-west)
    ax.fill_betweenx([-12, 12], -120, 200, color='#d4d4d4', alpha=0.4)
    ax.axhline(y=-12, color='gray', linewidth=0.5, linestyle='--')
    ax.axhline(y=12, color='gray', linewidth=0.5, linestyle='--')
    
    # Intersection highlight
    ax.fill_between([-12, 12], -12, 12, color='#ffeb3b', alpha=0.2)
    
    # Labels
    ax.text(-11, 85, 'TWY A', fontsize=9, color='gray', fontweight='bold')
    ax.text(150, 10, 'TWY B', fontsize=9, color='gray', fontweight='bold')
    ax.text(-10, -10, 'INTXN', fontsize=8, color='#f57f17', alpha=0.7)


def plot_airport_scenario(scenario, monitor):
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    gt = scenario['gt_trajectory']
    T = scenario['T']
    eta = monitor.eta if monitor.eta else 1.0
    
    titles = ['Hold-Short Plan (SAFE)', 'Proceed-Through Plan (UNSAFE)']
    plans = [scenario['safe_plan'], scenario['unsafe_plan']]
    
    for idx, (ax, plan, title) in enumerate(zip(axes, plans, titles)):
        draw_taxiway_layout(ax)
        ax.set_title(title, fontsize=13, fontweight='bold',
                    color='green' if 'SAFE' in title else 'red')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.15)
        ax.set_xlabel('x (meters)')
        ax.set_ylabel('y (meters)')
        
        # Draw FRS ellipses for selected timesteps
        for t_idx in [2, 5, 9, 14, 19]:
            if t_idx >= T:
                continue
            gmm = scenario['agent_gmms'][t_idx]
            c_star = solve_frs_convex_optimization(gmm, monitor.tau)
            for i, comp in enumerate(gmm.components):
                if c_star[i] > 0:
                    lbl = 'FRS (FORCE-OPT)' if t_idx == 2 and i == 0 else None
                    plot_ellipse(ax, comp.mean, comp.covariance, eta * c_star[i],
                               color='#00bcd4', alpha=0.15, label=lbl)
        
        # Contender ground truth
        ax.plot(*scenario['contender'].current_position, 'g^',
                markersize=12, label='Contender start')
        ax.plot(gt[:, 0], gt[:, 1], 'g.-', linewidth=2, markersize=5,
                label='Contender GT')
        
        # Ego plan
        pos = plan.positions
        color = '#1565C0' if idx == 0 else '#C62828'
        ax.plot(pos[:, 0], pos[:, 1], 'o-', color=color, linewidth=2.5,
                markersize=5, label=f'Ego plan')
        ax.plot(*plan.agent.current_position, 's', color=color,
                markersize=12, label='Ego start')
        
        # Hold-short line
        ax.axhline(y=-15, color='red', linewidth=1.5, linestyle=':',
                  alpha=0.7, label='Hold-short line')
        
        ax.legend(loc='upper left', fontsize=8)
        ax.set_xlim(-120, 200)
        ax.set_ylim(-140, 120)
    
    plt.suptitle('FORCE-OPT Aviation: Taxiway Intersection Safety Assessment',
                fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('aviation_force_opt.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_airport_scenario(scenario, monitor)

---
## 10. Bayesian Filter: In-Distribution vs. Unseen Airport

In [ ]:
def demo_ood_airport():
    """Show how the Bayesian filter responds to in-distribution (trained airport)
    vs. OOD (unseen airport) observations.
    
    This mirrors Amelia-TF's multi-airport experiments: training on e.g., KMDW
    and testing on KLAX (unseen), where predictions may be less reliable.
    """
    scenario = generate_airport_scenario(T=20, M=5)
    cal_data = generate_aviation_calibration_data(500, M=5)
    
    mon = AviationFORCE_OPT(tau=0.95, gamma=0.05, use_belief=True)
    mon.calibrate(cal_data, verbose=False)
    
    gt = scenario['gt_trajectory']
    
    # ID: contender follows predicted trajectory well
    bf_id = AviationBayesianFilter()
    for t in range(scenario['T']):
        bf_id.update(gt[t], scenario['agent_gmms'][t], mon.eta)
    
    # OOD: contender behaves very differently (e.g., unexpected go-around maneuver)
    bf_ood = AviationBayesianFilter()
    gt_ood = gt + np.array([0.0, 40.0])  # Large lateral offset
    for t in range(scenario['T']):
        bf_ood.update(gt_ood[t], scenario['agent_gmms'][t], mon.eta)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    steps = range(len(bf_id.history))
    ax.plot(steps, bf_id.history, 'g-o', linewidth=2, markersize=6,
            label='Trained airport (ID)')
    ax.plot(steps, bf_ood.history, 'r-s', linewidth=2, markersize=6,
            label='Unseen airport (OOD)')
    ax.axhline(y=0.75, color='orange', linestyle='--', linewidth=1.5,
              label='Tier 1 threshold (high conf)')
    ax.axhline(y=0.50, color='red', linestyle='--', linewidth=1.5,
              label='Tier 2 threshold (WC fallback)')
    ax.fill_between(steps, 0, 0.50, alpha=0.05, color='red')
    ax.fill_between(steps, 0.50, 0.75, alpha=0.03, color='orange')
    ax.set_xlabel('Timestep', fontsize=12)
    ax.set_ylabel('Expected β (model confidence)', fontsize=12)
    ax.set_title('Bayesian Confidence: Trained vs. Unseen Airport',
                fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    ax.text(18, 0.25, 'WC-FRS\nFallback', fontsize=9, ha='center', color='red', alpha=0.7)
    ax.text(18, 0.62, 'Inflated\nFRS', fontsize=9, ha='center', color='orange', alpha=0.7)
    plt.tight_layout()
    plt.savefig('aviation_bayesian_ood.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"ID final β̂={bf_id.history[-1]:.4f} (tier: {bf_id.confidence_tier})")
    print(f"OOD final β̂={bf_ood.history[-1]:.4f} (tier: {bf_ood.confidence_tier})")


demo_ood_airport()

---
## 11. Integration Guide: Connecting to Real Amelia-TF

Below is a reference showing how to connect this safety monitor to the actual Amelia-TF model.
This requires the Amelia framework to be installed (`pip install amelia_tf`).

In [ ]:
# ============================================================
# INTEGRATION REFERENCE (requires Amelia-TF installed)
# ============================================================
# This cell is reference code — it will not run without the full
# Amelia framework, dataset, and a trained checkpoint.
#
# Step 1: Load a pre-trained Amelia-TF model
# --------------------------------------------------
# import torch
# from amelia_tf.models import AmeliaTF  # Adjust import path as needed
# from amelia_tf.data import AmeliaDataModule
#
# ckpt_path = 'weights/Single-Airport/ksfo/checkpoints/best.ckpt'
# model = AmeliaTF.load_from_checkpoint(ckpt_path)
# model.eval()
#
# Step 2: Run inference on a scene
# --------------------------------------------------
# # The model outputs per-agent GMM parameters:
# #   mu:    (N_agents, M_modes, T_pred, 2)       mean positions
# #   sigma: (N_agents, M_modes, T_pred, 2, 2)    covariance matrices
# #   rho:   (N_agents, M_modes)                   mixing weights
#
# with torch.no_grad():
#     output = model(batch)  # batch from AmeliaDataModule
#     mu = output['mu'].cpu().numpy()      # (N, M, T, 2)
#     sigma = output['sigma'].cpu().numpy() # (N, M, T, 2, 2)
#     rho = output['rho'].cpu().numpy()     # (N, M)
#
# Step 3: Convert to FORCE-OPT format and evaluate
# --------------------------------------------------
# monitor = AviationFORCE_OPT(tau=0.95, gamma=0.05, use_belief=True)
#
# # Calibrate on validation split (do this once)
# cal_gmms = monitor.adapter.convert_batch(mu_cal, sigma_cal, rho_cal)
# cal_pairs = []
# for agent_idx in range(N_cal):
#     for t in range(T):
#         cal_pairs.append((cal_gmms[agent_idx][t], gt_positions_cal[agent_idx, t]))
# monitor.calibrate(cal_pairs)
#
# # At test time: evaluate ego plan
# agent_gmms_dict = {}
# for i, agent in enumerate(traffic_agents):
#     agent_gmms_dict[agent.agent_id] = monitor.adapter.convert_single_agent(
#         mu[i], sigma[i], rho[i]
#     )
#
# result = monitor.evaluate_plan_safety(
#     ego_plan=my_taxi_plan,
#     traffic_agents=traffic_agents,
#     traffic_gmms=agent_gmms_dict,
#     dt=1.0,
# )
# print(f"Plan is {'SAFE' if result['is_safe'] else 'UNSAFE'}")

print("Integration reference code (requires Amelia-TF framework to run).")
print("See comments in this cell for step-by-step instructions.")

---
## Summary

### What changed from the original FORCE-OPT:

| Component | Change | Rationale |
|---|---|---|
| `AmeliaTFAdapter` | **New** | Converts Amelia-TF's (mu, sigma, rho) tensors → `GMMDistribution` |
| `AgentType`, `ZoneType` | **New** | Aviation-specific agent and zone classifications |
| `SEPARATION_STANDARDS` | **New** | FAA/ICAO separation minima by zone type |
| `SurfacePlan` | **New** (replaces `MotionPlan`) | Zone-aware ego trajectory with separation lookups |
| `AviationBayesianFilter` | **Extended** | 3-tier confidence (high/medium/low) instead of binary |
| `compute_aviation_worst_case_frs` | **Replaced** | Taxiing aircraft dynamics instead of Dubins car |
| `evaluate_plan_safety` | **Extended** | Zone-dependent separation + multi-agent evaluation |
| Core optimization (Eq. 6) | **Unchanged** | Math is domain-agnostic (2D GMM → ellipsoids) |
| Conformal calibration | **Unchanged** | Same non-conformity scores and quantile method |

### What stayed the same:
- The convex optimization for FRS extraction (Theorem 2)
- The conformal prediction calibration procedure
- The Bayesian belief update equation (Eq. 13)
- The non-conformity score function (Eq. 7)

### To deploy on real airport data:
1. Install the Amelia framework and download the Amelia-48 dataset
2. Load a pre-trained Amelia-TF checkpoint (single or multi-airport)
3. Calibrate `AviationFORCE_OPT` on the validation split of your target airport(s)
4. At runtime, feed Amelia-TF predictions through `AmeliaTFAdapter` → `evaluate_plan_safety`